In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Employee contact directory

An HR export has mixed casing, one duplicate entry, some missing phone numbers, and a `full_name` column that should be split into two.

**New this week — `str.split(expand=True)`**

Splits each string on a separator and returns a DataFrame — one column per piece:

```python
df['full_name'].str.split(' ', expand=True)
#      0         1
# 0  Alice   Johnson
# 1    bob     smith

# Assign directly to new columns:
df[['first', 'last']] = df['full_name'].str.split(' ', expand=True)
```

After splitting, `.str.title()` capitalizes the first letter of each word — handy for names.

Clean the data (`yrs_exp` of `0` means unknown), then answer:
- How many employees remain after removing the duplicate?
- Add `first` and `last` columns. What are the first and last names of the employee with the most years of experience?
- What is the mean years of experience per department (excluding unknowns)? Use `np.nanmean` per group.
- Which department has the most employees?

In [20]:
contacts = pd.DataFrame({
    'emp_id':    ['E01','E02','E03','E04','E05','E06','E07','E08','E04'],
    'full_name': ['Alice Johnson','bob smith','CAROL WHITE','dave brown',
                  'EVE DAVIS','frank miller','GRACE LEE','henry clark','dave brown'],
    'dept':      ['Engineering','marketing','ENGINEERING','Sales',
                  'MARKETING','sales','Engineering','SALES','Sales'],
    'phone':     ['555-0101','555-0102','555-0103','n/a','555-0105',
                  'n/a','555-0107','555-0108','n/a'],
    'yrs_exp':   [5, 3, 8, 0, 6, 4, 7, 0, 0],
})

# Your code here

contacts = contacts.drop_duplicates()
print(contacts.shape[0],'remained')
contacts[['first','last']] = (contacts['full_name'].str.split(' ', expand = True))

contacts['first'] = contacts['first'].str.title()
contacts['last'] = contacts['last'].str.title()

print(contacts.loc[contacts['yrs_exp'].idxmax(),'first'],contacts.loc[contacts['yrs_exp'].idxmax(),'last'],'is the employee with max experience')

contacts['dept'] = contacts['dept'].str.lower()
g = contacts.groupby('dept')['yrs_exp'].apply(lambda x: np.nanmean(x))
print(g)

de = contacts['dept'].value_counts()
print(de.idxmax(),'has the most employees')

8 remained
Carol White is the employee with max experience
dept
engineering    6.666667
marketing      4.500000
sales          1.333333
Name: yrs_exp, dtype: float64
engineering has the most employees


---

## Level 2 — Retail shipment log

Ten shipments from a warehouse. The `destination` column stores city and state together (`"Austin, TX"`). The `weight_kg` column has `'N/A'` strings. The `delivered` column uses `1`/`0` codes.

Split `destination` into `city` and `state` columns — same technique as Level 1, but split on `', '`. Standardize both to lowercase. Then answer:

- Which city received the highest total shipment weight?
- What fraction of shipments were delivered? Replace `1`/`0` with `True`/`False` first, then use `np.mean`.
- What is the 75th percentile weight across valid shipments? Use `np.nanpercentile`.
- Which carrier handled the most shipments?

In [39]:
shipments = pd.DataFrame({
    'order_id':    ['S001','S002','S003','S004','S005',
                    'S006','S007','S008','S009','S010'],
    'destination': ['Austin, TX','portland, or','SEATTLE, WA','Denver, CO','austin, tx',
                    'PORTLAND, OR','Seattle, WA','DENVER, CO','Houston, TX','houston, tx'],
    'weight_kg':   [2.5, 'N/A', 4.1, 1.8, 2.5, 'N/A', 3.3, 1.8, 5.2, 5.2],
    'carrier':     ['FedEx','ups','FEDEX','UPS','fedex','UPS','Fedex','ups','FedEx','fedex'],
    'delivered':   [1, 1, 0, 1, 1, 1, 0, 1, 1, 1],
})

# Your code here

shipments[['city','state']] = shipments['destination'].str.split(', ',expand = True)
shipments['city'] = shipments['city'].str.lower()
shipments['state'] = shipments['state'].str.lower()
shipments['weight_kg'] = pd.to_numeric(shipments['weight_kg'], errors='coerce')

gg = shipments.groupby('city')['weight_kg'].apply(lambda x: np.nansum(x))

print(gg.idxmax(),'has the max total shipments weight')

shipments['delivered'] = shipments['delivered'].replace({1:True, 0:False})

print(shipments['delivered'].mean(),'has been delivered')
print(np.nanpercentile(shipments['weight_kg'],q = 75))
shipments['carrier'] = shipments['carrier'].str.lower()
print(shipments['carrier'].value_counts().idxmax(),'handle the most')

houston has the max total shipments weight
0.8 has been delivered
4.375
fedex handle the most


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_10746/2095831637.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  shipments['delivered'] = shipments['delivered'].replace({1:True, 0:False})


---

## Level 3 — Gym membership database

Sixteen records (one duplicate) from a gym's member system. Design the cleaning pipeline yourself — no function stubs provided.

Known issues:
- `member_id`: one exact duplicate
- `full_name`: mixed casing — split into `first` and `last` columns
- `plan`: inconsistent casing
- `fee`: dollar-sign strings; some entries are `'n/a'`
- `join_date`: some entries are `'TBD'`

Build a `.pipe()` chain. After cleaning:

1. Which plan has the most members?
2. Who logged the most checkins? Print their first and last name separately.
3. Log-transform valid fees with `np.log`. What is the standard deviation of the log-fees?
4. How many members joined before 2023? (Use `.dt.year` on `join_date`.)
5. What fraction of members are missing either their fee or their join date?

In [49]:
members = pd.DataFrame({
    'member_id': ['M001','M002','M003','M004','M005','M006','M007','M008',
                  'M009','M010','M011','M012','M013','M014','M015','M007'],
    'full_name': ['Alice Johnson','BOB SMITH','carol white','Dave Brown','EVE DAVIS',
                  'frank miller','GRACE LEE','henry clark','IVY CHEN','jack wilson',
                  'KAREN HALL','liam scott','MONA PATEL','noah kim','OLIVIA RYAN','GRACE LEE'],
    'plan':      ['gold','SILVER','bronze','GOLD','silver','Bronze','GOLD','silver',
                  'gold','BRONZE','Silver','gold','SILVER','bronze','GOLD','GOLD'],
    'fee':       ['$49','$29','$19','$49','$29','$19','$49','n/a',
                  '$49','$19','$29','$49','n/a','$19','$49','$49'],
    'join_date': ['2023-01-15','2023-02-22','2023-01-30','2022-12-01','2024-01-10',
                  '2023-06-15','2023-03-08','TBD','2022-11-20','2023-09-01',
                  '2024-02-14','2023-04-25','TBD','2023-07-11','2022-10-05','2023-03-08'],
    'checkins':  [48, 32, 15, 65, 28, 41, 71, 19, 53, 44, 55, 38, 22, 12, 62, 71],
})

# Your code here
def fil(df):
    df = df.copy()
    df = df.drop_duplicates()
    return df 

def convert(df):
    df[['first','last']] = df['full_name'].str.split(' ',expand = True)
    df['first'] = df['first'].str.title()
    df['last'] = df['last'].str.title()
    df['plan'] = df['plan'].str.title()
    df['fee'] = pd.to_numeric(df['fee'].str.replace('$','',regex = False), errors='coerce')
    df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce')
    return df



1. Which plan has the most members?
2. Who logged the most checkins? Print their first and last name separately.
3. Log-transform valid fees with `np.log`. What is the standard deviation of the log-fees?
4. How many members joined before 2023? (Use `.dt.year` on `join_date`.)
5. What fraction of members are missing either their fee or their join date?

In [64]:
clean = members.pipe(fil).pipe(convert)
print(clean['plan'].value_counts().idxmax(),'has the most members')
print(clean.loc[clean['checkins'].idxmax(),'first'], clean.loc[clean['checkins'].idxmax(),'last'], 'logged the most checkines')
clean['log_fee'] = np.log(clean['fee'])
print(np.nanstd(clean['log_fee']), 'is the standard deviation of log fee')
clean['join_year'] = clean['join_date'].dt.year
print((clean['join_year']<2023).sum(),'joined before 2023')
clean['stat'] = (pd.isna(clean['fee'])) |  (pd.isna(clean['join_date']))
print(clean['stat'].mean(),'of members are missing either thier fee or their join date')

Gold has the most members
Grace Lee logged the most checkines
0.4116550103617177 is the standard deviation of log fee
3 joined before 2023
0.13333333333333333 of members are missing either thier fee or their join date
